# Differentiable Pulsetable Synthesis for Wind Instrument Modeling

Link to the paper: https://ieeexplore.ieee.org/abstract/document/11462505

In this notebook, we show how a trained model can be loaded and recreate some figures from the paper.

Author: Simon Schwär <mail@simon-schwaer.de><br>
Date: October 2025 

---

Before we can load our experiment based on the operative config that is saved in the results folder, we load some libraries and recreate the entrypoint of libdamp.

In [ ]:
import glob
import os
import sys

import gin
import IPython.display as ipd
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

gin.enter_interactive_mode()


@gin.configurable("libdamp")
def libdamp_entry_point(experiment=gin.REQUIRED, train_dataset=gin.REQUIRED, val_dataset=None, test_dataset=None):
    """Entrance function for specifying an experiment and dataset for training in the gin config

    Example usage in the gin config:
    ```
    torchdsp.experiment = @MyModel
    torchdsp.train_dataset = @MyTrainDataset
    torchdsp.val_dataset = @MyValDataset
    torchdsp.test_dataset = @MyTestDataset
    ```

    Validation and test datasets are optional.
    Attention: The function name conflicts with the module name, but this is only relevant in this file.
    """
    return experiment, train_dataset, val_dataset, test_dataset

Now we can load the model. `exp_id` is the experiment identifier and corresponds to the folder name in `results`. The sampling rate should mat

In [ ]:
fs = 48000.0
exp_id = "icassp2026_20260820_172625"

sys.path.append("../..")

# load operative gin config
gin.clear_config()
with gin.unlock_config():
    gin.parse_config_file(os.path.join("../../results", exp_id, "operative_config.gin"), skip_unknown=True)

    gin.bind_parameter("train/PulseItDataset.path", "../../../datasets/ChoraleBricks/libdamp_all")
    gin.bind_parameter("val/PulseItDataset.path", "../../../datasets/ChoraleBricks/libdamp_all")
    gin.bind_parameter("test/PulseItDataset.path", "../../../datasets/ChoraleBricks/libdamp_all")

# create model and datasets
model, train_dataset, val_dataset, test_dataset = libdamp_entry_point()

best_ckpt = glob.glob(os.path.join("../../results", exp_id, "checkpoints/*best*.ckpt"))[0]
last_ckpt = os.path.join("../../results", exp_id, "checkpoints/last.ckpt")

# load the best checkpoint and configure the weights of the model
ckpt = torch.load(best_ckpt, map_location="cpu")
sd = ckpt["state_dict"]

model.load_state_dict(sd)
# model = model.cuda()
_ = model.eval()

assert model.fs == fs, "Sampling rate mismatch"

Separately from the model, we can load the training metadata that is stored in `results/<exp_id>/csv_log/metrics.csv` and visualize the loss over time.

In [ ]:
# plot training and validation loss
steps_per_epoch = 141  # for calculating the epoch average

log = pd.read_csv(os.path.join("../../results", exp_id, "csv_log/metrics.csv"))
train_loss = log[log["train_loss_step"].notnull()]
train_loss_epoch = log[log["train_loss_epoch"].notnull()]
val_loss = log[log["val_loss"].notnull()]

idx = train_loss_epoch["train_loss_epoch"].argmin()
print(f"Best train loss: {train_loss_epoch['train_loss_epoch'].iloc[idx]:.4f} (epoch {train_loss_epoch['step'].iloc[idx] / steps_per_epoch:.0f})")
print(f"Best val loss: {val_loss['val_loss'].min():.4f}, last val los: {val_loss['val_loss'].iloc[-1]:.4f}")

plt.plot(train_loss["step"], train_loss["train_loss_step"], label="train")
plt.plot(train_loss_epoch["step"] - 0.5 * steps_per_epoch, train_loss_epoch["train_loss_epoch"], label="train (epoch average)")
plt.plot(val_loss["step"], val_loss["val_loss"], label="validation")
plt.legend()
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

First, let's visualize the learned pulses...

In [ ]:
pulses = model.synth.pulsetable.pw_table.cpu().detach().numpy()[0]
pulses /= np.max(np.abs(pulses))


def lighten_hex_color(c: str, amount: float) -> str:
    c = c.lstrip("#")

    r = int(c[0:2], 16)
    g = int(c[2:4], 16)
    b = int(c[4:6], 16)

    if amount <= 0.5:
        # interpolate from black to the color
        t = amount / 0.5
        r = int(round(r * t))
        g = int(round(g * t))
        b = int(round(b * t))
    else:
        # interpolate from the color to white
        t = (amount - 0.5) / 0.5
        r = int(round(r + (255 - r) * t))
        g = int(round(g + (255 - g) * t))
        b = int(round(b + (255 - b) * t))

    return f"#{r:02X}{g:02X}{b:02X}"


color = "#1C7221"  # "#AD5D00" #

t_pulse = np.arange(pulses.shape[1]) / fs

_, axs = plt.subplots(1, 2, figsize=(8.5, 2.5), dpi=100, tight_layout=True, gridspec_kw={"width_ratios": [3, 4]})

for i in range(pulses.shape[0]):
    axs[0].plot(t_pulse * 1000, pulses[i], c=lighten_hex_color(color, i / 6 + 0.3), lw=2)
axs[0].set_ylim(-1.25, 1.25)
axs[0].set_xlim(-0.05, t_pulse[-1] * 1000 + 0.05)
axs[0].set_xlabel("Time (ms)")
axs[0].set_ylabel("Amplitude")
axs[0].set_title("Time domain")

L_pad = 4096
f_sig = np.fft.rfftfreq(L_pad, 1 / fs)
for i in range(pulses.shape[0]):
    P = np.abs(np.fft.rfft(np.pad(pulses[i], (0, L_pad - len(pulses[i]))))) / len(pulses[i]) * 2
    axs[1].plot(f_sig / 1000, 20 * np.log10(P), c=lighten_hex_color(color, i / 6 + 0.3))
axs[1].set_ylim(-60, 3)
axs[1].set_xlim(-0.05, 5.6)
axs[1].set_xlabel("Frequency (kHz)")
axs[1].set_ylabel("Mag. (dB)")
axs[1].set_title("Frequency domain")

plt.tight_layout(h_pad=0.5)

plt.show()

... and the learned post filter.

In [ ]:
# analyze post-filter
h_post = model.post_filter.filter.cpu().detach()[0, 0]
L_post = h_post.shape[0]

_, axs = plt.subplots(2, 1, figsize=(8.5, 5), dpi=100, tight_layout=True)

axs[0].plot(h_post, c="k", lw=2)
axs[0].set_xlabel("Sample")
axs[0].set_ylabel("Amplitude")
axs[0].set_title("Time domain")

f_post = np.fft.rfftfreq(L_post, 1 / fs)
H_post = np.abs(np.fft.rfft(h_post)) / L_post * 2
axs[1].plot(f_post, 20 * np.log10(P), c="k")
axs[1].set_ylim(-60, 3)
axs[1].set_xlim(20, fs / 2)
axs[1].set_xscale("log")
axs[1].set_xlabel("Frequency (Hz)")
axs[1].set_ylabel("Mag. (dB)")
axs[1].set_title("Frequency domain")

plt.show()

To check if the model works, let's try some samples from the test dataset. Note that in the default configuration, the test datasets corresponds to the *different voice* (DV) condition as described in the paper, i.e., the F0 likely lies below the range seen during training.

In [ ]:
F = 128  # frame length for control parameters

for idx in np.random.choice(len(test_dataset), 2):
    print(idx)
    x, f0, _, g = test_dataset[idx]
    x = x.unsqueeze(0)
    f0 = f0.unsqueeze(0)
    g = g.unsqueeze(0)

    y, w = model(x, f0, g, return_weight=True)
    y = y.detach()
    w = w.detach()

    g_est = model.estimate_gain(x).detach()

    H = 1024
    N = 4096
    gamma = 10

    X = np.abs(librosa.stft(x[0].numpy(), n_fft=N, hop_length=H))
    Y = np.abs(librosa.stft(y[0].numpy(), n_fft=N, hop_length=H))

    t_fr = np.arange(g_est.shape[1]) * F / fs
    t_ = np.arange(X.shape[1]) * H / fs
    f_ = np.arange(X.shape[0]) * fs / N

    _, axs = plt.subplots(4, 1, figsize=(4.75, 3.5), dpi=300, sharex=True, gridspec_kw={"height_ratios": [4, 1.33, 2.0, 4]})

    axs[0].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * X), cmap="gray_r")
    axs[0].set_ylim(0.05, 2.5)
    axs[0].set_ylabel("Freq. (kHz)")

    axs[1].plot(t_fr, g_est[0].detach().numpy(), c="k", lw=1.5)
    axs[1].set_ylim(0, 1.1)
    axs[1].set_ylabel("Gain")

    axs[2].pcolormesh(t_fr, np.arange(w.shape[-1]) + 1, w[0].T, cmap="gray_r")
    axs[2].set_ylabel("Pulse")
    axs[2].set_yticks([])

    axs[3].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * Y), cmap="gray_r")
    axs[3].set_ylim(0.05, 2.5)
    axs[3].set_ylabel("Freq. (kHz)")
    axs[3].set_xlabel("Time (s)")

    plt.tight_layout(h_pad=0.3)

    plt.show()

    print("Reference:")
    ipd.display(ipd.Audio(np.repeat(x.numpy(), 2, axis=0), rate=fs))
    print("Model output:")
    ipd.display(ipd.Audio(np.repeat(y.numpy(), 2, axis=0), rate=fs))

Finally, we can now process arbitrary input audio.

In [ ]:
# load audio
x, _ = librosa.load("audio/trumpet.wav", sr=fs)

# estimate F0 (we are using the pYIN implementation of librosa here for convenience, but CREPE, PESTO, or any other method would also work)
f0, f0_voiced, _ = librosa.pyin(x, fmin=200, fmax=750, sr=fs, frame_length=2048, hop_length=F)

# convert to Tensors in the right shape
x = torch.Tensor(x)[None, :]
f0 = torch.Tensor(f0)[None, :-1]

# run model
y, w = model(x, f0, None, return_weight=True)
y = y.detach()
w = w.detach()
g_est = model.estimate_gain(x).detach()

# visualize
H = 1024
N = 4096
gamma = 10

X = np.abs(librosa.stft(x[0].numpy(), n_fft=N, hop_length=H))
Y = np.abs(librosa.stft(y[0].numpy(), n_fft=N, hop_length=H))

t_fr = np.arange(g_est.shape[1]) * F / fs
t_ = np.arange(X.shape[1]) * H / fs
f_ = np.arange(X.shape[0]) * fs / N

_, axs = plt.subplots(4, 1, figsize=(4.75, 3.5), dpi=300, sharex=True, gridspec_kw={"height_ratios": [4, 1.33, 2.0, 4]})

axs[0].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * X), cmap="gray_r")
axs[0].set_ylim(0.05, 2.5)
axs[0].set_ylabel("Freq. (kHz)")

axs[1].plot(t_fr, g_est[0].detach().numpy(), c="k", lw=1.5)
axs[1].set_ylim(0, 1.1)
axs[1].set_ylabel("Gain")

axs[2].pcolormesh(t_fr, np.arange(w.shape[-1]) + 1, w[0].T, cmap="gray_r")
axs[2].set_ylabel("Pulse")
axs[2].set_yticks([])

axs[3].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * Y), cmap="gray_r")
axs[3].set_ylim(0.05, 2.5)
axs[3].set_ylabel("Freq. (kHz)")
axs[3].set_xlabel("Time (s)")

plt.tight_layout(h_pad=0.3)

plt.show()

print("Reference:")
ipd.display(ipd.Audio(np.repeat(x.numpy(), 2, axis=0), rate=fs))
print("Model output:")
ipd.display(ipd.Audio(np.repeat(y.numpy(), 2, axis=0), rate=fs))